# AlignLab — Experiment 04: Base vs SFT vs DPO

Evaluate **Qwen/Qwen3-1.7B** on held-out UltraFeedback Binarized preference pairs. The SFT and DPO adapters are independently applied to the same quantized base model. **No retraining** occurs.

Metric: per-response mean conditional log-probability (chosen versus rejected), evaluated on the *same prompts*. Higher chosen-minus-rejected margin is better. This is a small benchmark, not a general model-quality guarantee.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])

In [ ]:
%pip install -q \
    "transformers>=4.51,<5" \
    "peft>=0.15,<0.19" \
    "accelerate>=1.4,<2" \
    "datasets>=3.5,<5" \
    "bitsandbytes>=0.45" \
    "huggingface-hub>=0.30,<1" \
    sentencepiece

In [ ]:
import json
import time
from datetime import UTC, datetime
from pathlib import Path

import torch
from datasets import load_dataset
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), "Enable a GPU accelerator."
assert torch.cuda.device_count() == 1, "Restart the runtime with one visible GPU."
MODEL_ID = "Qwen/Qwen3-1.7B"
SFT_REPO = "hamedready/alignlab-qwen3-1.7b-qlora"
DPO_REPO = "hamedready/alignlab-qwen3-1.7b-dpo"
DATASET_ID = "trl-lib/ultrafeedback_binarized"
SEED = 42
EVAL_START = 16  # DPO training used test shuffle(seed=42)[:16] for validation.
EVAL_COUNT = 40
MAX_LENGTH = 512
OUT = (
    Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path("/content")
    if Path("/content").exists()
    else Path(".")
)
OUTPUT_DIR = OUT / "alignlab-three-way"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0), "Output:", OUTPUT_DIR)

## Adapter discovery

On Kaggle, attach both private adapter datasets: `hamedhani/alignlab-qlora-adapter` and `hamedhani/alignlab-dpo-adapter`. On Colab, store a read-capable `HF_TOKEN` in Secrets. No secrets are needed inside Kaggle.

In [ ]:
# Kaggle may mount datasets under /kaggle/input/datasets/<owner>/<slug>.
if Path("/kaggle/input").exists():
    print("Kaggle input directories:")
    for config_path in sorted(Path("/kaggle/input").rglob("adapter_config.json")):
        print("Adapter:", config_path.parent)


def resolve_adapter(slug, repo):
    root = Path("/kaggle/input")
    if root.exists():
        matches = sorted(root.rglob("adapter_config.json"))
        matches = [p.parent for p in matches if slug in p.parts]
        if len(matches) != 1:
            raise FileNotFoundError(
                f"Expected exactly one {slug} adapter; found {matches}. "
                "Check Kaggle dataset_sources."
            )
        folder = matches[0]
        if not any(
            (folder / name).is_file() for name in ("adapter_model.safetensors", "adapter_model.bin")
        ):
            raise FileNotFoundError(f"Adapter weights missing in {folder}")
        return str(folder), None
    try:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")
    except ImportError:
        token = os.environ.get("HF_TOKEN")
    if not token:
        raise RuntimeError("HF_TOKEN required for private Hub adapters outside Kaggle")
    return repo, token.strip()


sft_source, sft_token = resolve_adapter("alignlab-qlora-adapter", SFT_REPO)
dpo_source, dpo_token = resolve_adapter("alignlab-dpo-adapter", DPO_REPO)
print("SFT source:", sft_source)
print("DPO source:", dpo_source)

## Held-out preference pairs

Experiment 03 used 80 training examples from the `train` split and 16 validation examples from the shuffled `test` split. We evaluate indices **16–55** of that same shuffled `test` split to avoid overlap with the earlier DPO validation subset. This separation assumes Experiment 03's dataset revision and sampling remain unchanged.

In [ ]:
raw = load_dataset(DATASET_ID, split="test")
assert {"chosen", "rejected"}.issubset(raw.column_names)
raw = raw.shuffle(seed=SEED)
assert len(raw) >= EVAL_START + EVAL_COUNT
heldout = raw.select(range(EVAL_START, EVAL_START + EVAL_COUNT))


def parse_pair(row):
    chosen = row["chosen"]
    rejected = row["rejected"]
    if not isinstance(chosen, list) or not isinstance(rejected, list):
        raise ValueError("Expected conversational chosen/rejected messages")
    prompt = [m for m in chosen if m.get("role") != "assistant"]
    chosen_reply = next(
        (m["content"] for m in reversed(chosen) if m.get("role") == "assistant"), ""
    )
    rejected_reply = next(
        (m["content"] for m in reversed(rejected) if m.get("role") == "assistant"), ""
    )
    if not prompt or not chosen_reply or not rejected_reply:
        raise ValueError("Empty prompt or response in preference example")
    return {"prompt": prompt, "chosen": chosen_reply, "rejected": rejected_reply}


pairs = [parse_pair(row) for row in heldout]
print("Held-out preference pairs:", len(pairs))

## Load base model and both adapters

Both adapters were trained from the original base model; the exported DPO adapter was initialized with the SFT weights. Do **not** stack SFT and DPO. Use `disable_adapter()` for the base and `set_adapter()` for each fine-tuned model.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quant, device_map={"": 0}, dtype=torch.float16
)
model = PeftModel.from_pretrained(
    base, sft_source, adapter_name="sft", is_trainable=False, token=sft_token
)
model.load_adapter(dpo_source, adapter_name="dpo", is_trainable=False, token=dpo_token)
model.eval()
print("Loaded adapters:", list(model.peft_config))

## Conditional likelihood scoring

Score only completion tokens, excluding the prompt. Normalize by scored completion tokens to reduce length bias. The model's ranking can still be influenced by response length and formatting. Each model sees identical tokenized examples.

In [ ]:
def make_inputs(prompt_messages, response):
    prefix = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    # Use the same assistant-turn boundary for chosen and rejected responses.
    completion = response + (tokenizer.eos_token or "")
    prompt_ids = tokenizer(prefix, add_special_tokens=False)["input_ids"]
    completion_ids = tokenizer(completion, add_special_tokens=False)["input_ids"]
    budget = MAX_LENGTH - len(completion_ids)
    if budget < 1:
        completion_ids = completion_ids[: MAX_LENGTH - 1]
        budget = 1
    prompt_ids = prompt_ids[-budget:]
    ids = prompt_ids + completion_ids
    labels = [-100] * len(prompt_ids) + completion_ids
    if len(completion_ids) < 2:
        raise ValueError("Completion too short for scoring")
    return (torch.tensor([ids], device="cuda:0"), torch.tensor([labels], device="cuda:0"))


def mean_completion_logprob(prompt, response):
    input_ids, labels = make_inputs(prompt, response)
    with torch.inference_mode():
        logits = model(input_ids=input_ids, use_cache=False).logits
        log_probs = torch.log_softmax(logits[:, :-1].float(), dim=-1)
        targets = labels[:, 1:]
        mask = targets.ne(-100)
        token_logprobs = log_probs.gather(-1, targets.clamp_min(0).unsqueeze(-1)).squeeze(-1)
        return float(token_logprobs[mask].mean().item())


def evaluate_active_adapter():
    rows = []
    start = time.perf_counter()
    for index, pair in enumerate(pairs):
        chosen = mean_completion_logprob(pair["prompt"], pair["chosen"])
        rejected = mean_completion_logprob(pair["prompt"], pair["rejected"])
        rows.append(
            {
                "index": index,
                "chosen_logprob": chosen,
                "rejected_logprob": rejected,
                "margin": chosen - rejected,
                "correct": chosen > rejected,
            }
        )
        if (index + 1) % 10 == 0:
            print(f"Scored {index + 1}/{len(pairs)}")
    return {
        "preference_accuracy": sum(x["correct"] for x in rows) / len(rows),
        "mean_margin": sum(x["margin"] for x in rows) / len(rows),
        "mean_chosen_logprob": sum(x["chosen_logprob"] for x in rows) / len(rows),
        "mean_rejected_logprob": sum(x["rejected_logprob"] for x in rows) / len(rows),
        "evaluation_seconds": time.perf_counter() - start,
        "per_example": rows,
    }

In [ ]:
print("Evaluating base model...")
with model.disable_adapter():
    base_results = evaluate_active_adapter()
print("Evaluating SFT adapter...")
model.set_adapter("sft")
sft_results = evaluate_active_adapter()
print("Evaluating DPO adapter...")
model.set_adapter("dpo")
dpo_results = evaluate_active_adapter()
for name, result in (("base", base_results), ("sft", sft_results), ("dpo", dpo_results)):
    print(
        name,
        "accuracy:",
        round(result["preference_accuracy"], 3),
        "margin:",
        round(result["mean_margin"], 4),
    )

## Save reproducible report

Preference accuracy is an automatic proxy, not human judgment. A small evaluation set has considerable uncertainty. All three models use the same quantization and preference pairs; model execution order can still influence timing.

In [ ]:
import peft
import transformers

report = {
    "experiment": "04_three_way_evaluation",
    "timestamp_utc": datetime.now(UTC).isoformat(),
    "base_model": MODEL_ID,
    "sft_adapter": SFT_REPO,
    "dpo_adapter": DPO_REPO,
    "dataset": DATASET_ID,
    "split": "test",
    "selection": f"shuffle(seed={SEED})[{EVAL_START}:{EVAL_START + EVAL_COUNT}]",
    "examples": len(pairs),
    "max_length": MAX_LENGTH,
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "peft_version": peft.__version__,
    "scoring": "mean conditional completion log probability per token",
    "base": base_results,
    "sft": sft_results,
    "dpo": dpo_results,
}
report_path = OUTPUT_DIR / "three_way_comparison.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Saved:", report_path)

In [ ]:
import shutil

archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Archive:", archive)

## Interpretation

Compare DPO vs SFT **preference accuracy** and mean margin first. The DPO model may improve preference ranking without reducing reference-text NLL. Before making broader quality claims, use a larger held-out benchmark, confidence intervals, and blinded human or judge-based assessments.